# Prototyp 1 – Word2Vec: Statische Wortvektoren & Wortnachbarschaften

**Projekt:** Vom Wort zum Vektor – Embeddings für Texte  
**Korpus:** Leipzig Corpora Collection (`deu_news_2010_100K`, deutsche Nachrichten 2010, 100 000 Sätze)  
**Modell:** Word2Vec (Gensim, **Skip-Gram**) – selbst trainiert

---

## Forschungsfrage

Welche semantisch oder thematisch plausiblen Wortnachbarschaften entstehen in einem
selbst trainierten Word2Vec-Modell auf einem **begrenzten** deutschsprachigen Korpus?

Das Notebook bildet den reproduzierbaren CLI-Ablauf (`python -m src.word2vec.pipeline`)
Schritt für Schritt nach und stellt jeden Verarbeitungsschritt anschaulich dar.

## Ablauf

| # | Schritt |
|---|---|
| 1 | Imports & Konfiguration |
| 2 | Konzept & Mini-Demonstration (Tokenisierung + Skip-Gram-Fenster) |
| 3 | Korpus herunterladen & einlesen |
| 4 | Vorverarbeitung: Bereinigung, Tokenisierung, Statistik |
| 5 | Training des Word2Vec-Modells |
| 6 | Erste Nachbarschaftsabfrage am Modell |
| 7 | Nachbarschaftsanalyse aller Zielwörter |
| 8 | Visualisierung: PCA-Projektion |
| 9 | Fachliche Einordnung der Nachbarschaften |
| 10 | Zusammenfassung & Interpretation |

## 1 · Imports & Konfiguration

Das Notebook liegt in `notebooks/`. Damit `src.word2vec` importierbar ist, wird der
Projektroot zu `sys.path` hinzugefügt. Als Kernel die `.venv` (Python 3.13) wählen.
Die Konfiguration stammt vollständig aus `configs/word2vec.yaml` (Single Source of Truth).

In [ ]:
%matplotlib inline
import sys, json, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

# Notebook liegt in notebooks/ – Projektroot eine Ebene höher
PROJECT_ROOT = Path('../').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.word2vec.config import (
    CORPUS_ID, CORPUS_URL, CORPUS_PAGE,
    WORD2VEC_PARAMS, TARGET_WORDS,
    DEFAULT_SAMPLE_SIZE, DEFAULT_SEED, DEFAULT_MIN_SENTENCE_TOKENS, DEFAULT_TOPN,
    ARCHIVE_PATH, SENTENCES_JSONL, PREPARE_METADATA_PATH,
    MODEL_PATH, MODEL_METADATA_PATH,
    NEIGHBORS_CSV, NEIGHBORS_MD, TARGETS_JSON, PCA_FIGURE, TARGET_FIGURE_DIR,
    OUTPUT_TABLES_DIR, OUTPUT_FIGURES_DIR,
)
from src.word2vec.corpus import (
    iter_sentences_from_archive, clean_text, tokenize,
    sample_sentences, tokenize_sentences, corpus_stats, save_tokenized_sentences,
)
from src.word2vec.evaluate import resolve_target_words, build_neighbor_rows, write_markdown
from src.word2vec.visualize import plot_target_pcas

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Projektverzeichnis: {PROJECT_ROOT}')
print(f'Korpus-ID:          {CORPUS_ID}')
print(f'Korpus-Quelle:      {CORPUS_PAGE}')

In [ ]:
print('Word2Vec-Hyperparameter (configs/word2vec.yaml → word2vec_params):')
for k, v in WORD2VEC_PARAMS.items():
    print(f'  {k:12s}: {v}')

voll = DEFAULT_SAMPLE_SIZE >= 100_000
print(f'\nStichprobe:                  {"vollständiger 100K-Korpus" if voll else DEFAULT_SAMPLE_SIZE}')
print(f'Mindest-Satzlänge:           {DEFAULT_MIN_SENTENCE_TOKENS} Tokens')
print(f'Nachbarn je Zielwort (topn): {DEFAULT_TOPN}')

print(f'\n{len(TARGET_WORDS)} Zielwörter (mit Ersatzwörtern als Fallback):')
for word, fallbacks in TARGET_WORDS.items():
    print(f'  {word:12s} → {", ".join(fallbacks)}')

## 2 · Konzept & Mini-Demonstration

**Wie lernt Word2Vec?**

Word2Vec lernt **statische** Wortvektoren aus dem Kontext: Wörter, die in ähnlichen
Umgebungen vorkommen, erhalten ähnliche Vektoren (*distributionelle Hypothese* –
„You shall know a word by the company it keeps“). Wir verwenden die
**Skip-Gram**-Variante (`sg=1`): Aus dem Zielwort werden die umliegenden
Kontextwörter innerhalb eines Fensters (`window=5`) vorhergesagt.

Anders als bei SBERT gibt es **kein vortrainiertes Modell** – die Vektoren werden
hier selbst auf dem Leipzig-Korpus trainiert. Die Ähnlichkeit zweier Wortvektoren
$\mathbf{u}, \mathbf{v}$ messen wir – wie bei SBERT – über die **Kosinusähnlichkeit**:

$$\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert\mathbf{u}\rVert \, \lVert\mathbf{v}\rVert}$$

Bevor trainiert werden kann, muss der Rohtext in Token-Listen zerlegt werden.
Das demonstrieren wir an **einem** Beispielsatz und zeigen die Skip-Gram-Kontextpaare.

In [ ]:
demo_sentence = 'Die Bundesregierung plant 2010 neue Investitionen in den Arbeitsmarkt.'
demo_tokens = tokenize(demo_sentence)

print('Rohsatz:    ', demo_sentence)
print('Bereinigt:  ', clean_text(demo_sentence))
print('Tokens:     ', demo_tokens)
print(f'→ {len(demo_tokens)} Tokens: kleingeschrieben, ohne Satzzeichen/Zahlen, Umlaute erhalten')

# Skip-Gram-Kontextpaare (Zielwort -> Kontextwort) zur Veranschaulichung
window = 2
center = 'bundesregierung'
ci = demo_tokens.index(center)
context = [demo_tokens[j] for j in range(max(0, ci - window), min(len(demo_tokens), ci + window + 1)) if j != ci]
print(f"\nSkip-Gram-Trainingspaare (window={window}) für das Zielwort '{center}':")
for c in context:
    print(f'  ({center}  →  {c})')

## 3 · Korpus herunterladen & einlesen

Der Korpus **`deu_news_2010_100K`** der [Leipzig Corpora Collection](https://corpora.wortschatz-leipzig.de/)
ist ein deutscher Nachrichten-Teilkorpus aus dem Jahr 2010 mit 100 000 Sätzen.
Die Sätze liegen im Format `Sentence_ID<TAB>Satztext` vor.

Der Download erfolgt nur, falls das Archiv noch nicht in `data/raw/` liegt
(die Funktion `download_corpus()` prüft das selbst).

In [ ]:
from src.word2vec.download import download_corpus

download_corpus()  # lädt nur herunter, falls das Archiv noch nicht vorhanden ist

raw_sentences = list(iter_sentences_from_archive(ARCHIVE_PATH))
print(f'\nEingelesene Rohsätze: {len(raw_sentences):,}'.replace(',', '.'))
print('\nDrei Beispielsätze aus dem Korpus:')
for s in raw_sentences[:3]:
    print(f'  • {s}')

## 4 · Vorverarbeitung: Bereinigung, Tokenisierung & Statistik

Jeder Satz wird bereinigt (HTML-Reste, BOM, doppelte Leerzeichen), kleingeschrieben
und auf Wortebene tokenisiert. Tokens sind zusammenhängende Buchstabenfolgen
(`[a-zäöüß]+`) – Satzzeichen und Zahlen werden verworfen, deutsche Umlaute bleiben
erhalten. Sätze mit weniger als `min_sentence_tokens` Tokens werden entfernt.

Da `sample_size: null` gesetzt ist, wird der **vollständige** 100K-Korpus verwendet
(feste, reproduzierbare Datenbasis). Die tokenisierten Sätze und Metadaten werden –
wie in `src/word2vec/prepare.py` – als JSONL gespeichert.

In [ ]:
sampled   = sample_sentences(raw_sentences, sample_size=DEFAULT_SAMPLE_SIZE, seed=DEFAULT_SEED)
tokenized = tokenize_sentences(sampled, min_sentence_tokens=DEFAULT_MIN_SENTENCE_TOKENS)

stats = corpus_stats(tokenized)
print('Korpus-Statistik nach der Tokenisierung:')
for key, value in stats.items():
    print(f'  {key:22s}: {value:,}'.replace(',', '.'))

# Tokenisierte Sätze + Metadaten speichern (wie src/word2vec/prepare.py)
save_tokenized_sentences(tokenized, SENTENCES_JSONL)
prepare_metadata = {
    'corpus_id': CORPUS_ID, 'corpus_page': CORPUS_PAGE,
    'source_sentences': len(raw_sentences),
    'selection_method': 'full_corpus' if DEFAULT_SAMPLE_SIZE >= len(raw_sentences) else 'random_sample',
    'selected_sentences': len(sampled),
    'seed': DEFAULT_SEED, 'min_sentence_tokens': DEFAULT_MIN_SENTENCE_TOKENS,
    **stats,
}
PREPARE_METADATA_PATH.write_text(json.dumps(prepare_metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'\nGespeichert: {SENTENCES_JSONL.name}  &  {PREPARE_METADATA_PATH.name}')

### 4a · Häufigkeitsverteilung der Tokens

Natürliche Sprache folgt dem **Zipf'schen Gesetz**: wenige Wörter sind extrem häufig,
sehr viele Wörter sehr selten. Der Parameter `min_count=5` verwirft die seltene
„lange Schwanz"-Region, weil für solche Wörter keine stabilen Vektoren lernbar sind.

In [ ]:
counts       = Counter(t for s in tokenized for t in s)
sent_lengths = [len(s) for s in tokenized]
most_common  = counts.most_common(20)
mc           = WORD2VEC_PARAMS['min_count']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) 20 häufigste Tokens
words, freqs = zip(*most_common)
axes[0].barh(range(len(words)), freqs, color='#276fbf')
axes[0].set_yticks(range(len(words))); axes[0].set_yticklabels(words, fontsize=8)
axes[0].invert_yaxis(); axes[0].set_title('20 häufigste Tokens', fontsize=10)
axes[0].set_xlabel('Häufigkeit')

# (b) Satzlängen-Verteilung
axes[1].hist(sent_lengths, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].axvline(np.mean(sent_lengths), color='#e74c3c', linestyle='--',
                label=f'Ø {np.mean(sent_lengths):.1f} Tokens')
axes[1].set_title('Satzlängen (Tokens je Satz)', fontsize=10)
axes[1].set_xlabel('Tokens je Satz'); axes[1].set_ylabel('Anzahl Sätze')
axes[1].set_xlim(0, 60); axes[1].legend(fontsize=8)

# (c) Zipf-Verteilung (log-log)
sorted_freqs = sorted(counts.values(), reverse=True)
axes[2].loglog(range(1, len(sorted_freqs) + 1), sorted_freqs, color='#8e44ad')
axes[2].axhline(mc, color='#e74c3c', linestyle='--', label=f'min_count = {mc}')
axes[2].set_title('Zipf-Verteilung (Rang vs. Häufigkeit)', fontsize=10)
axes[2].set_xlabel('Rang'); axes[2].set_ylabel('Häufigkeit'); axes[2].legend(fontsize=8)

fig.suptitle('Korpus-Charakteristik nach der Tokenisierung', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / 'word2vec_corpus_stats.png', dpi=150, bbox_inches='tight')
plt.show()

below = sum(1 for v in counts.values() if v < mc)
print(f'Wörter unterhalb min_count={mc}: {below:,} von {len(counts):,} '
      f'({below/len(counts):.1%}) werden beim Training verworfen.'.replace(',', '.'))

## 5 · Training des Word2Vec-Modells

Trainiert wird mit der dokumentierten **Basiskonfiguration** (nicht als Optimum gedacht):

| Parameter | Wert | Bedeutung |
|---|---:|---|
| `vector_size` | 100 | Dimension der Wortvektoren |
| `window` | 5 | Kontextfenster links/rechts |
| `min_count` | 5 | Mindesthäufigkeit eines Wortes |
| `sg` | 1 | Skip-Gram (statt CBOW) |
| `epochs` | 10 | Trainingsdurchläufe |
| `workers` | 1 | stabilere Reproduzierbarkeit |
| `seed` | 42 | Reproduzierbarkeit |

`min_count` reduziert das Vokabular von der rohen Tokenzahl auf die tatsächlich
modellierten Wörter.

In [ ]:
from gensim.models import Word2Vec

print('Trainiere Word2Vec (Skip-Gram) – das dauert typischerweise rund eine Minute ...')
t0 = time.perf_counter()
model = Word2Vec(sentences=tokenized, **WORD2VEC_PARAMS)
runtime = round(time.perf_counter() - t0, 2)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(str(MODEL_PATH))

model_metadata = {
    'runtime_seconds': runtime,
    'sentence_count': len(tokenized),
    'token_count': stats['tokens'],
    'raw_vocabulary_size': stats['raw_vocabulary_size'],
    'final_vocabulary_size': len(model.wv),
    'parameters': dict(WORD2VEC_PARAMS),
}
MODEL_METADATA_PATH.write_text(json.dumps(model_metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'\nTraining abgeschlossen in {runtime:.1f} s')
print(f'Vokabular roh (alle Tokentypen):           {stats["raw_vocabulary_size"]:,}'.replace(',', '.'))
print(f'Vokabular im Modell (min_count={WORD2VEC_PARAMS["min_count"]} gefiltert): {len(model.wv):,}'.replace(',', '.'))
print(f'Vektordimension:                           {model.wv.vector_size}')
print(f'Modell gespeichert: {MODEL_PATH.name}')

## 6 · Erste Nachbarschaftsabfrage am Modell

Ein kurzer „Funktionstest": Wie sieht ein Wortvektor aus, und welche Nachbarn liefert
das Modell für ein einzelnes Wort? `most_similar` rankt alle Vokabular-Wörter nach
Kosinusähnlichkeit zum Zielwort.

In [ ]:
probe = 'regierung'
vec = model.wv[probe]
print(f"Wortvektor für '{probe}':  shape {vec.shape},  erste 8 Komponenten:")
print(' ', np.round(vec[:8], 3))

print(f"\n5 nächste Nachbarn von '{probe}':")
for rank, (nb, sim) in enumerate(model.wv.most_similar(probe, topn=5), 1):
    print(f'  {rank}. {nb:18s} {sim:.4f}')

print('\nDirekte Kosinusähnlichkeit zweier Wortpaare:')
print(f"  regierung ↔ bundesregierung: {model.wv.similarity('regierung', 'bundesregierung'):.4f}  (verwandt)")
print(f"  regierung ↔ software:        {model.wv.similarity('regierung', 'software'):.4f}  (unverwandt)")

## 7 · Nachbarschaftsanalyse aller Zielwörter

Für jedes konfigurierte Zielwort werden die `topn=5` nächsten Nachbarn bestimmt.
Fehlt ein Zielwort im Vokabular, greift `resolve_target_words` auf das erste
verfügbare Ersatzwort zurück. Das Ergebnis wird als CSV **und** Markdown gespeichert
(identisch zu `src/word2vec/evaluate.py` → `write_csv` / `write_markdown`).

In [ ]:
resolved_targets = resolve_target_words(model)
missing = sorted(set(TARGET_WORDS) - set(resolved_targets))
print(f'Im Vokabular aufgelöste Zielwörter: {len(resolved_targets)}/{len(TARGET_WORDS)}')
if missing:
    print(f'Nicht gefunden (kein Ersatzwort im Vokabular): {missing}')
else:
    print('Alle Zielwörter sind direkt im Vokabular vorhanden – kein Fallback nötig.')

rows = build_neighbor_rows(model, resolved_targets, topn=DEFAULT_TOPN)
neighbors = pd.DataFrame(rows)
neighbors.to_csv(NEIGHBORS_CSV, index=False, encoding='utf-8')
write_markdown(rows, NEIGHBORS_MD)  # zusätzlich als Markdown (identisch zu src/word2vec/evaluate.py)
print(f'\n{len(neighbors)} Nachbarschaftszeilen gespeichert → {NEIGHBORS_CSV.name} & {NEIGHBORS_MD.name}')

In [ ]:
# Kompakte Übersicht: je Zielwort die fünf nächsten Nachbarn in einer Zeile
rows_by_target = {}
for r in rows:
    rows_by_target.setdefault(r['angefragtes_zielwort'], []).append(r)

overview = pd.DataFrame([
    {'Zielwort': req,
     'Nächste Nachbarn (Kosinusähnlichkeit)':
         ', '.join(f"{x['nachbar']} ({x['cosine_similarity']:.3f})" for x in rows_by_target[req])}
    for req in resolved_targets
])
pd.set_option('display.max_colwidth', 130)
overview

## 8 · Visualisierung: PCA-Projektion

Die 100-dimensionalen Vektoren der Zielwörter und ihrer Nachbarn werden mit
**PCA** auf zwei Dimensionen reduziert. Wörter, die im hochdimensionalen Raum nahe
beieinander liegen, erscheinen tendenziell auch in der Projektion nahe beieinander.

> **Hinweis:** Die PCA ist eine stark vereinfachte 2D-Projektion eines
> 100D-Raums und dient ausschließlich der explorativen Veranschaulichung –
> nicht als eigenständiger Leistungsnachweis.

In [ ]:
def collect_words(subset):
    '''Sammelt Zielwörter und Nachbarn in stabiler Reihenfolge (wie visualize.plot_pca).'''
    out = []
    for r in subset:
        for col in ('verwendetes_zielwort', 'nachbar'):
            w = str(r[col])
            if w not in out and w in model.wv:
                out.append(w)
    return out

target_used = {req: rows_by_target[req][0]['verwendetes_zielwort'] for req in resolved_targets}
used_set = set(target_used.values())

# jedem Wort sein (angefragtes) Zielwort zuordnen, für die Einfärbung
target_of = {}
for r in rows:
    target_of.setdefault(str(r['verwendetes_zielwort']), str(r['angefragtes_zielwort']))
    target_of.setdefault(str(r['nachbar']), str(r['angefragtes_zielwort']))

words   = collect_words(rows)
vectors = np.array([model.wv[w] for w in words])
coords  = PCA(n_components=2, random_state=42).fit_transform(vectors)

targets_list = list(resolved_targets.keys())
cmap = plt.cm.tab10
color_for = {t: cmap(i % 10) for i, t in enumerate(targets_list)}

fig, ax = plt.subplots(figsize=(13, 9))
for w, (x, y) in zip(words, coords):
    is_target = w in used_set
    col = color_for.get(target_of.get(w), '#999999')
    ax.scatter(x, y, s=110 if is_target else 38, color=col,
               edgecolor='black' if is_target else 'none',
               linewidth=1.1, zorder=3 if is_target else 2)
    ax.annotate(w, (x, y), fontsize=10 if is_target else 8,
                fontweight='bold' if is_target else 'normal', alpha=0.9)

handles = [mpatches.Patch(color=color_for[t], label=t) for t in targets_list]
ax.legend(handles=handles, fontsize=8, ncol=2, title='Zielwort-Cluster', loc='best')
ax.set_title('PCA-Projektion: Zielwörter (umrandet) und ihre nächsten Nachbarn', fontsize=12)
ax.set_xlabel('PCA-Komponente 1'); ax.set_ylabel('PCA-Komponente 2')
plt.tight_layout()
# Dateiname an das Modul angeglichen (config.PCA_FIGURE → word2vec_pca_neighbors.png)
plt.savefig(PCA_FIGURE, dpi=150, bbox_inches='tight')
plt.show()

### 8a · Einzelne Zielwörter im Detail

Pro Zielwort eine eigene Mini-Projektion (Zielwort **rot** + fünf Nachbarn).
So lassen sich die einzelnen „Bedeutungsnachbarschaften" direkt vergleichen.

Zusätzlich werden – über `visualize.plot_target_pcas` – die **modul-kompatiblen
Einzelgrafiken** (`word2vec_targets/pca_*.png`) und die Zielwort-Metadaten
(`word2vec_target_words.json`) geschrieben, sodass das Notebook exakt dieselben
Artefakte wie `python -m src.word2vec.pipeline` erzeugt.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, req in zip(axes.flat, targets_list):
    subset = rows_by_target[req]
    used = target_used[req]
    wlist = collect_words(subset)
    cc = PCA(n_components=2, random_state=42).fit_transform(np.array([model.wv[w] for w in wlist]))
    for w, (x, y) in zip(wlist, cc):
        is_t = (w == used)
        ax.scatter(x, y, s=70 if is_t else 36, color='#c0392b' if is_t else '#276fbf', zorder=3 if is_t else 2)
        ax.annotate(w, (x, y), fontsize=8, fontweight='bold' if is_t else 'normal',
                    color='#c0392b' if is_t else 'black')
    ax.set_title(f"'{used}'", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('PCA je Zielwort: Zielwort (rot) + 5 nächste Nachbarn', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / 'word2vec_targets_grid.png', dpi=140, bbox_inches='tight')
plt.show()

# Zusätzlich die modul-kompatiblen Einzelgrafiken je Zielwort erzeugen
# (identisch zu src/word2vec/evaluate.py → outputs/figures/word2vec_targets/pca_*.png).
# plot_pca erzwingt das Agg-Backend; daher das Inline-Backend danach wiederherstellen,
# damit nachfolgende Zellen weiterhin inline rendern.
_prev_backend = matplotlib.get_backend()
target_figures = plot_target_pcas(model, rows, TARGET_FIGURE_DIR)
plt.switch_backend(_prev_backend)

# Zielwort-Metadaten schreiben (identisch zu src/word2vec/evaluate.py → word2vec_target_words.json)
target_info = {
    'resolved_targets': resolved_targets,
    'missing_targets': missing,
    'target_figures': target_figures,
    'topn': DEFAULT_TOPN,
}
TARGETS_JSON.write_text(json.dumps(target_info, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Modul-kompatible Artefakte gespeichert: {TARGETS_JSON.name} und '
      f'{len(target_figures)} Einzelgrafiken in {TARGET_FIGURE_DIR.name}/')

## 9 · Fachliche Einordnung der Nachbarschaften

Word2Vec-Nachbarn sind **nicht automatisch Synonyme**. Für die Seminararbeit ist
entscheidend, *welche Art* von Nähe das Modell erfasst. Wir ordnen jede
Nachbarschaft in drei Gruppen ein:

| Gruppe | Bedeutung | Farbe |
|---|---|---|
| **semantisch plausibel** | ähnliche Bedeutung / Ober-/Unterbegriff | grün |
| **thematisch plausibel** | gleiches Themenfeld, aber keine Bedeutungsgleichheit | orange |
| **problematisch** | irreführende Nähe: Eigennamen-Fragmente, falscher Wortsinn, Kollokations-Artefakte | rot |

Die Einordnung ist eine **manuelle fachliche** Bewertung (im Code-Modul als Platzhalter
`"manuell einordnen"` vorgesehen).

In [ ]:
# Manuelle Einordnung der Nachbarn, Schlüssel: (angefragtes_zielwort, nachbar)
EINORDNUNG = {
    ('regierung', 'bundesregierung'): 'semantisch plausibel',
    ('regierung', 'opposition'):      'thematisch plausibel',
    ('regierung', 'teheran'):         'problematisch',
    ('regierung', 'erdogan'):         'problematisch',
    ('regierung', 'armee'):           'thematisch plausibel',
    ('wahl', 'bundespräsidenten'):    'thematisch plausibel',
    ('wahl', 'kandidatur'):           'thematisch plausibel',
    ('wahl', 'bundesversammlung'):    'thematisch plausibel',
    ('wahl', 'wulff'):                'problematisch',
    ('wahl', 'jochimsen'):            'problematisch',
    ('partei', 'opposition'):           'thematisch plausibel',
    ('partei', 'fraktion'):             'thematisch plausibel',
    ('partei', 'cdu'):                  'semantisch plausibel',
    ('partei', 'linkspartei'):          'semantisch plausibel',
    ('partei', 'regierungskoalition'):  'thematisch plausibel',
    ('markt', 'wettbewerber'):  'thematisch plausibel',
    ('markt', 'arbeitsmarkt'):  'semantisch plausibel',
    ('markt', 'handel'):        'thematisch plausibel',
    ('markt', 'sektor'):        'thematisch plausibel',
    ('markt', 'preisen'):       'thematisch plausibel',
    ('unternehmen', 'konzern'):       'semantisch plausibel',
    ('unternehmen', 'firmen'):        'semantisch plausibel',
    ('unternehmen', 'versicherer'):   'thematisch plausibel',
    ('unternehmen', 'sap'):           'problematisch',
    ('unternehmen', 'positionierung'):'problematisch',
    ('bank', 'irish'):       'problematisch',
    ('bank', 'bhf'):         'thematisch plausibel',
    ('bank', 'scotland'):    'problematisch',
    ('bank', 'commerzbank'): 'semantisch plausibel',
    ('bank', 'merrill'):     'problematisch',
    ('daten', 'passwörter'):  'thematisch plausibel',
    ('daten', 'websites'):    'thematisch plausibel',
    ('daten', 'bankkunden'):  'thematisch plausibel',
    ('daten', 'gesammelten'): 'problematisch',
    ('daten', 'html'):        'thematisch plausibel',
    ('software', 'ios'):         'thematisch plausibel',
    ('software', 'betaversion'): 'thematisch plausibel',
    ('software', 'client'):      'thematisch plausibel',
    ('software', 'nvidia'):      'problematisch',
    ('software', 'apps'):        'semantisch plausibel',
    ('modell', 'feature'):    'thematisch plausibel',
    ('modell', 'design'):     'thematisch plausibel',
    ('modell', 'kaufsignal'): 'problematisch',
    ('modell', 'system'):     'semantisch plausibel',
    ('modell', 'tool'):       'thematisch plausibel',
    ('netz', 'gedächtnis'):   'problematisch',
    ('netz', 'eingespeist'):  'thematisch plausibel',
    ('netz', 'telefonieren'): 'thematisch plausibel',
    ('netz', 'brett'):        'problematisch',
    ('netz', 'server'):       'thematisch plausibel',
}

FALLBACK = 'thematisch plausibel'
neighbors['einordnung'] = [
    EINORDNUNG.get((r['angefragtes_zielwort'], r['nachbar']), FALLBACK) for r in rows
]
n_fallback = sum(1 for r in rows if (r['angefragtes_zielwort'], r['nachbar']) not in EINORDNUNG)
print(f'Eingeordnete Nachbarn: {len(neighbors)}  |  ohne kuratiertes Label (Fallback): {n_fallback}')
print('\nVerteilung der Einordnung:')
print(neighbors['einordnung'].value_counts().to_string())

In [ ]:
order  = ['semantisch plausibel', 'thematisch plausibel', 'problematisch']
colors = {'semantisch plausibel': '#27ae60', 'thematisch plausibel': '#f39c12', 'problematisch': '#e74c3c'}

pivot = (neighbors.groupby(['angefragtes_zielwort', 'einordnung']).size()
         .unstack(fill_value=0).reindex(columns=order, fill_value=0)
         .reindex(index=list(resolved_targets.keys())))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [2, 1]})

pivot.plot(kind='barh', stacked=True, color=[colors[c] for c in order], ax=ax1,
           edgecolor='white', linewidth=0.6)
ax1.set_title('Einordnung der Nachbarn je Zielwort', fontsize=11)
ax1.set_xlabel('Anzahl Nachbarn'); ax1.set_ylabel('')
ax1.invert_yaxis(); ax1.legend(fontsize=8, loc='lower right')

totals = neighbors['einordnung'].value_counts().reindex(order, fill_value=0)
ax2.bar(range(len(order)), totals.values, color=[colors[c] for c in order])
ax2.set_xticks(range(len(order)))
ax2.set_xticklabels(['semantisch', 'thematisch', 'problematisch'], rotation=12, fontsize=9)
ax2.set_title(f'Gesamtverteilung (n = {int(totals.sum())})', fontsize=11)
ax2.set_ylabel('Anzahl Nachbarn')
for i, v in enumerate(totals.values):
    ax2.text(i, v + 0.3, str(int(v)), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES_DIR / 'word2vec_neighbor_classification.png', dpi=150, bbox_inches='tight')
plt.show()

### 9a · Die problematischen Fälle

Die problematischen Nachbarschaften zeigen die zentralen **Grenzen** statischer
Wortvektoren auf einem kleinen Korpus:

- **Eigennamen-Dominanz / NER-Leakage:** `bank` → *irish, scotland, merrill* sind
  Fragmente von Bank-Eigennamen (*Anglo Irish Bank*, *Royal Bank of Scotland*,
  *Merrill Lynch*) statt des Konzepts „Bank". Ebenso `regierung` → *teheran, erdogan*.
- **Polysemie (ein Vektor pro Wort):** `netz` mischt Stromnetz (*eingespeist*),
  Telefonnetz (*telefonieren*), Internet (*server*) und neuronales/Gedächtnis-Netz –
  Word2Vec kann diese Lesarten **nicht** trennen.
- **Korpus-Spezifika 2010:** `wahl` → *wulff, jochimsen* spiegelt die
  Bundespräsidentenwahl 2010 – thematisch erklärbar, aber nicht generalisierbar.

In [ ]:
prob = (neighbors[neighbors['einordnung'] == 'problematisch']
        [['angefragtes_zielwort', 'verwendetes_zielwort', 'nachbar', 'cosine_similarity']]
        .reset_index(drop=True))
print(f'{len(prob)} als „problematisch" eingeordnete Nachbarschaften:\n')
print(prob.to_string(index=False))

## 10 · Zusammenfassung & Interpretation

In [ ]:
summary = pd.DataFrame([
    {'Kennzahl': 'Korpus',                       'Wert': f'{CORPUS_ID}'},
    {'Kennzahl': 'Sätze (Korpus)',               'Wert': f'{len(raw_sentences):,}'.replace(',', '.')},
    {'Kennzahl': 'Tokens (gesamt)',              'Wert': f"{stats['tokens']:,}".replace(',', '.')},
    {'Kennzahl': 'Vokabular (roh)',              'Wert': f"{stats['raw_vocabulary_size']:,}".replace(',', '.')},
    {'Kennzahl': f'Vokabular (Modell, min_count={WORD2VEC_PARAMS["min_count"]})',
                                                 'Wert': f'{len(model.wv):,}'.replace(',', '.')},
    {'Kennzahl': 'Vektordimension',              'Wert': str(model.wv.vector_size)},
    {'Kennzahl': 'Trainingszeit',                'Wert': f'{runtime:.1f} s'},
    {'Kennzahl': 'Zielwörter im Vokabular',      'Wert': f'{len(resolved_targets)}/{len(TARGET_WORDS)}'},
    {'Kennzahl': 'Analysierte Nachbarn',         'Wert': f'{len(neighbors)} ({DEFAULT_TOPN} je Zielwort)'},
    {'Kennzahl': 'davon semantisch / thematisch / problematisch',
                 'Wert': ' / '.join(str(int(totals[c])) for c in order)},
])
summary

### Interpretation für die wissenschaftliche Arbeit

**Stärken:**
- Aus einem **begrenzten** Korpus (100 000 Sätze, ~1,6 Mio. Tokens) entstehen
  Wortvektoren, deren Nachbarschaften überwiegend **thematisch kohärent** sind
  (Politik, Wirtschaft, Technik bilden erkennbare Cluster – siehe PCA).
- Echte **semantische** Nachbarn treten auf (`unternehmen` → *konzern, firmen*;
  `partei` → *cdu, linkspartei*; `markt` → *arbeitsmarkt*).

**Schwächen:**
- **Polysemie:** Ein Wort = ein Vektor. Mehrdeutige Wörter (`netz`, `bank`, `modell`)
  vermischen ihre Lesarten und erzeugen die meisten problematischen Nachbarn.
- **Eigennamen-Dominanz:** Häufige Eigennamen aus den 2010er-Nachrichten
  (*teheran, erdogan, wulff, scotland*) verdrängen abstraktere Nachbarn.
- **Korpus- & Parameterabhängigkeit:** Die Ergebnisse gelten nur unter dieser
  Datenbasis und Basiskonfiguration; sie sind kein allgemeines Urteil über Word2Vec.

**Fazit:**
Word2Vec macht aus reiner Ko-Okkurrenz plausible Wort­nachbarschaften sichtbar und
illustriert das Prinzip statischer Embeddings eindrücklich. Gleichzeitig zeigen die
problematischen Fälle die prinzipielle Grenze auf, die **kontextsensitive** Modelle
wie SBERT (Prototyp 2) motivieren: Dort wird Bedeutung nicht mehr pro Wort, sondern
**pro Kontext** kodiert.